

## Introduction to Hugging Face

Hugging Face provides **state-of-the-art** NLP (and increasingly, multimodal) models and an easy-to-use Python library called **Transformers**. With just a few lines of code you can:

* **Load** pre-trained models for dozens of tasks
* **Run** inference via high-level “pipelines”
* **Fine-tune** on your own data

---

## 1. Setup in Google Colab

```bash
# Install the core libraries
!pip install transformers datasets huggingface_hub --quiet
```

> **Tip:** Colab often comes with a GPU—go to **Runtime → Change runtime type → GPU** for faster inference.

---

## 2. Quick-start with Pipelines

The `pipeline` API wraps tokenization, model loading, and inference in one object.

### 2.1 Sentiment Analysis

```python
from transformers import pipeline

# 1. Load a sentiment-analysis pipeline (defaults to a small model)
sentiment = pipeline("sentiment-analysis")

# 2. Run inference
examples = [
    "Hugging Face makes NLP super accessible!",
    "I dislike bugs in my code..."
]
results = sentiment(examples)
for text, res in zip(examples, results):
    print(f"{text!r:50} → label={res['label']}, score={res['score']:.3f}")
```

### 2.2 Text Generation

```python
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
prompt = "Once upon a time"
out = generator(prompt, max_length=30, num_return_sequences=1)
print(out[0]["generated_text"])
```

### 2.3 Question Answering

```python
from transformers import pipeline

qa = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")
context = (
    "Hugging Face is an AI company with the mission to democratize good machine learning. "
    "The Transformers library provides thousands of pretrained models in 100+ languages."
)

answer = qa({
    "question": "What is the mission of Hugging Face?",
    "context": context
})
print(answer["answer"])
```

### 2.4 Model Discovery with `huggingface_hub`

```python
from huggingface_hub import HfApi

api = HfApi()
# List top 5 English summarization models
models = api.list_models(task="summarization", limit=5)
for m in models:
    print(m.modelId)
```

---

## 3. Assignment: Build Your Own Pipeline

**Your task:**

1. **Choose** one problem from the list below.
2. **Find** a suitable pre-trained model on [Hugging Face Models](https://huggingface.co/models).
3. **Create** a `pipeline` in Colab to solve the problem—and demonstrate it on 2–3 examples.
4. **Experiment** with model parameters (e.g. `max_length`, `top_k`, `temperature`) and **compare** results.

### 🔹 Problem Set

* **Sentiment classification** of product reviews
* **Text summarization** of news articles
* **Machine translation** (e.g., English ↔ French)
* **Named entity recognition** on a text snippet
* **Paraphrasing** a given sentence
* **Any other** pipeline-supported task you’re curious about!

---



In [12]:
!pip install -U transformers sentencepiece --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 107.5 MB/s eta 0:00:00


In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "humarin/chatgpt_paraphraser_on_T5_base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


def paraphrase(sentence, max_length=128, top_k=50, temperature=1.0):
    prompt = f"paraphrase: {sentence} </s>"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        top_k=top_k,
        temperature=temperature,
        num_return_sequences=1
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


# Test examples
examples = [
    "Artificial intelligence is changing the way people work.",
    "The students completed their project before the deadline.",
    "Learning Python can help you build many useful applications."
]

for sentence in examples:
    print("Original   :", sentence)
    print("Paraphrase :", paraphrase(sentence))
    print("-" * 80)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Original   : Artificial intelligence is changing the way people work.
Paraphrase : People's work is being transformed by the implementation of AI.
--------------------------------------------------------------------------------
Original   : The students completed their project before the deadline.
Paraphrase : The students met their requirement as soon as possible.
--------------------------------------------------------------------------------
Original   : Learning Python can help you build many useful applications.
Paraphrase : Python is a valuable language to learn for building practical applications.
--------------------------------------------------------------------------------


In [14]:
sentence = "Artificial intelligence is changing the way people work."

settings = [
    {"name": "Conservative", "top_k": 20, "temperature": 0.7},
    {"name": "Balanced", "top_k": 50, "temperature": 1.0},
    {"name": "Creative", "top_k": 100, "temperature": 1.3},
]

print("Original:", sentence)
print("=" * 80)

for setting in settings:

    result = paraphrase(
        sentence,
        max_length=128,
        top_k=setting["top_k"],
        temperature=setting["temperature"]
    )

    print(
        f"{setting['name']} "
        f"(top_k={setting['top_k']}, "
        f"temperature={setting['temperature']}):"
    )

    print(result)
    print("-" * 80)

Original: Artificial intelligence is changing the way people work.
Conservative (top_k=20, temperature=0.7):
The nature of human work is being reshaped by artificial intelligence.
--------------------------------------------------------------------------------
Balanced (top_k=50, temperature=1.0):
Artificial Intelligence is making work different for people.
--------------------------------------------------------------------------------
Creative (top_k=100, temperature=1.3):
people who’re in the middle of altering their work to some degree under artificial intelligence.
--------------------------------------------------------------------------------
